In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## Breast Cancer Dataset

In [12]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=10):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], max_depth=3, epochs=10),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.998
DecisionTree: 0.998
NDT: 0.962


## Iris Dataset

In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_iris()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=4):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], max_depth=3, epochs=10),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 953us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 966us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.995
DecisionTree: 1.000
NDT: 0.999


## Wine Dataset

In [4]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = load_wine()
X = data.data
y = data.target
feature_names = data.feature_names
class_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict_proba(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=False,
    mode='classification',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=13):
    explanations = []

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], max_depth=3, epochs=10),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")


mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
mean_leaf_values shape: (8, 1, 1)
self.L: 8 self.C: 1
mean_leaf_values shape after squeeze: (8, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
=== Stabilité des explications (LIME avec modèles locaux différents) ===
LinearRegression: 0.999
DecisionTree: 0.997
NDT: 0.999


## California Housing Dataset

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# 0. Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Créer de faux coef_ et intercept_ pour compatibilité LIME
        importances = self.feature_importances_
        self.coef_ = np.array(importances)
        self.intercept_ = 0
        return self

# ========================
# 1. Charger dataset
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction pour tester la stabilité
# ========================
def explanation_stability(explainer, model_regressor, n_repeats=5, instance_id=0, num_features=8):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]
    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=model_regressor
        )

        # transformer l’explication en vecteur importance
        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # similarité cosinus moyenne
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Comparer modèles locaux
# ========================
results = {}
# LinearRegression avec explainer classique
results["LinearRegression"] = explanation_stability(explainer_classic, LinearRegression(), n_repeats=5, instance_id=0)
# DecisionTree (wrapper) avec explainer NDT
results["DecisionTree"] = explanation_stability(explainer_ndt, DecisionTreeWrapper(), n_repeats=5, instance_id=0)
# Neural Decision Tree avec explainer NDT
results["NDT"] = explanation_stability(
    explainer_ndt,
    NDTRegressorWrapper(D=X_train.shape[1], max_depth=3, epochs=10),
    n_repeats=5,
    instance_id=0,
    num_features=X_train.shape[1]  # obligatoire pour NDT
)

print("=== Stabilité des explications (LIME avec modèles locaux différents) ===")
for name, score in results.items():
    print(f"{name}: {score:.3f}")